### Cleaning and Validating Inventory Snapshots

In [0]:
# Reading Bronze Table
df_bronze = spark.readStream.format("delta").load(
"/Volumes/mini_project_team_a3t7/default/mini_project/bronze/store_inventory_stream"
)

In [0]:
# Deduplication
from pyspark.sql import functions as F

df_dedup = df_bronze.dropDuplicates(
    ["store_id", "product_id", "_ingestion_timestamp"]
)

In [0]:
# Data Type Enforcement
df_standardized = (
    df_dedup
    .withColumnRenamed("current_quantity", "inventory_quantity")
)

In [0]:
# Data Type Enforcement
df_casted = (
    df_standardized
    .withColumn("inventory_quantity", F.col("inventory_quantity").cast("int"))
)

In [0]:
# Buissness Rules
valid_store = F.col("store_id").isNotNull()
valid_product = F.col("product_id").isNotNull()
valid_quantity = F.col("inventory_quantity") >= 0

df_valid = df_casted.filter(
    valid_store & valid_product & valid_quantity
)

In [0]:
# Writing in silver layer

query = (
    df_valid.writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        "/Volumes/mini_project_team_a3t7/default/mini_project/checkpoint/silver/inventory_snapshots/"
    )
    .trigger(availableNow=True)
    .start(
        "/Volumes/mini_project_team_a3t7/default/mini_project/silver/inventory_snapshots/"
    )
)


In [0]:
query = (
    df_valid.writeStream
        .format("delta")
        .outputMode("append")
        .option(
            "checkpointLocation",
            "/Volumes/mini_project_team_a3t7/default/mini_project/checkpoint/silver/inventory_snapshots/"
        )
        .trigger(availableNow=True)
        .toTable("mini_project_team_a3t7.silver.inventory_snapshots")
)